# Raster algebra & apply

Run computations over raster values:

- **`apply(func)`** — apply a NumPy function to a band, returning a new `Dataset`.
- **`map_blocks(func, tile_size=...)`** — apply a function tile-by-tile (memory-friendly).
- **`overlay(classes_map)`** — group a raster's values by the classes in a second raster.

(For lazy, Dask-backed algebra see the [Dask quickstart](../dask/dataset.ipynb).)

## Setup

In [1]:
import os

os.environ['MPLBACKEND'] = 'Agg'

import tempfile
from pathlib import Path

import numpy as np


def _find_data():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / 'tests' / 'data'
        if cand.is_dir():
            return cand.resolve()
    raise FileNotFoundError('Could not locate tests/data from ' + str(Path.cwd()))


DATA = _find_data()
WORK = Path(tempfile.mkdtemp(prefix='pyramids-t3-'))
DATA.is_dir(), WORK.is_dir()

(True, True)

In [2]:
from pyramids.dataset import Dataset

ds = Dataset.read_file(str(DATA / 'acc4000.tif'))
ds.shape

2026-06-08 23:58:33 | INFO | pyramids.base.config | Logging is configured.


(1, 13, 14)

## Element-wise — `apply`

Apply a function to every cell of a band; returns a new `Dataset`.

In [3]:
doubled = ds.apply(lambda a: a * 2)
type(doubled).__name__, float(np.nanmax(np.where(
    doubled.read_array() == doubled.no_data_value[0], np.nan, doubled.read_array())))

('Dataset', 176.0)

## Tile-by-tile — `map_blocks`

Process the raster in `tile_size` blocks — the same result, a bounded memory footprint.

In [4]:
sen = Dataset.read_file(str(DATA / 'geotiff' / 'sentinel_crop.tif'))
incremented = sen.map_blocks(lambda a: a + 1, tile_size=128)
type(incremented).__name__, incremented.shape

('Dataset', (1, 256, 256))

## Group by class — `overlay`

`overlay` reads a second raster of class labels and returns a dict mapping each class value to
the list of data-raster values that fall in it — a raster-on-raster zonal summary.

In [5]:
classes = Dataset.read_file(str(DATA / 'geotiff' / 'sentinel-classes.tif'))
grouped = sen.overlay(classes)
type(grouped).__name__, sorted(grouped), [len(v) for v in grouped.values()]

('dict', [np.uint16(2)], [5771])

## Notes

- `apply(..., inplace=True)` mutates the dataset; `overlay` takes `exclude_value=` to drop a
  sentinel before grouping.
- See also: [Zonal statistics](zonal-statistics.ipynb) (the vector-polygon equivalent).